# Regime Detection

Fit HMM on real macro data. Do regimes meaningfully separate signal performance?

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

from src.data_bridge import load_all_signal_inputs
from src.regime import fit_regime_model, regime_summary
from src.signal_diagnostics import regime_ic, compute_ic_timeseries, ic_summary

## Load real macro data

In [ ]:
macro = pd.read_parquet("../data/raw/macro_panel.parquet")
macro.index = pd.to_datetime(macro.index)
print(f"Macro panel: {macro.shape}  ({macro.index[0].date()} to {macro.index[-1].date()})")
print(macro.describe().round(2))

## 2-regime HMM

In [ ]:
model2, labels2, scaler2 = fit_regime_model(macro, n_regimes=2)

summary2 = regime_summary(macro, labels2)
print("Regime summary (2 regimes):")
print(summary2.xs("mean", axis=1, level=1).round(2).to_string())
print(f"\nRegime counts:\n{labels2.value_counts().sort_index().to_string()}")

## Regime labels over time + VIX

In [ ]:
def plot_regimes(macro, labels, title):
    colors = {0: "lightsteelblue", 1: "lightsalmon", 2: "lightgreen"}
    labels_desc = {0: "Regime 0 (calm/low-VIX)", 1: "Regime 1 (stressed/high-VIX)", 2: "Regime 2"}

    fig, ax = plt.subplots(figsize=(14, 4))

    # Color background by contiguous regime blocks
    prev_regime = labels.iloc[0]
    block_start = labels.index[0]
    for date, regime in labels.items():
        if regime != prev_regime:
            ax.axvspan(block_start, date, alpha=0.4,
                       color=colors[prev_regime], linewidth=0)
            block_start = date
            prev_regime = regime
    ax.axvspan(block_start, labels.index[-1], alpha=0.4,
               color=colors[prev_regime], linewidth=0)

    ax.plot(macro.index, macro["vix"], color="black", linewidth=1.2, label="VIX")
    ax.set_ylabel("VIX")
    ax.set_title(title)

    n_regimes = labels.nunique()
    patches = [mpatches.Patch(color=colors[r], alpha=0.5, label=labels_desc[r])
               for r in range(n_regimes)]
    ax.legend(handles=patches + [plt.Line2D([0],[0], color="black", label="VIX")],
              fontsize=8)
    plt.tight_layout()
    plt.show()

plot_regimes(macro, labels2, "2-regime HMM labels vs VIX (real macro data)")

## IC per regime — all signals

In [ ]:
# Load signal panel and rebuild outcome
panel = pd.read_parquet("../data/processed/signal_panel.parquet")
panel["date"] = pd.to_datetime(panel["date"])
sig_cols = [c for c in panel.columns if c not in ("ticker", "date")]

data    = load_all_signal_inputs(n_tickers=100, start="2012-01-01", end="2024-12-31")
prices  = data["prices"]
wide    = prices.pivot(index="date", columns="ticker", values="adj_close")
qprices = wide.resample("QE").last()
fwd     = qprices.shift(-1) / qprices - 1
outcome_df = fwd.stack().reset_index()
outcome_df.columns = ["date", "ticker", "outcome"]
outcome_df = outcome_df.dropna()

quarter_ends = set(outcome_df["date"].unique())
panel_q = panel[panel["date"].isin(quarter_ends)].copy()

print(f"Signal panel at quarter-ends: {panel_q.shape}")

In [ ]:
# regime_ic() aligns quarterly IC dates with monthly regime labels
# (quarter-ends are a subset of month-ends, so intersection works)
rows = []
for col in sig_cols:
    sig_df = panel_q[["ticker", "date", col]].dropna().rename(columns={col: "signal"})
    if len(sig_df) < 100:
        continue
    try:
        ric = regime_ic(sig_df, outcome_df, labels2)
        row = {"signal": col}
        for _, r in ric.iterrows():
            row[f"regime_{int(r['regime'])}_ic"] = r["mean_ic"]
            row[f"regime_{int(r['regime'])}_n"] = int(r["n_periods"])
        rows.append(row)
    except Exception as e:
        print(f"  WARNING: {col} failed — {e}")

ric_df = pd.DataFrame(rows)
ric_df["ic_spread"] = ric_df["regime_1_ic"] - ric_df["regime_0_ic"]
ric_df = ric_df.sort_values("ic_spread", key=abs, ascending=False)

fmt = {"regime_0_ic": "{:+.3f}".format, "regime_1_ic": "{:+.3f}".format,
       "ic_spread": "{:+.3f}".format}
print("\nIC per regime (sorted by |spread|):")
print(ric_df[["signal","regime_0_ic","regime_0_n","regime_1_ic","regime_1_n","ic_spread"]]
      .to_string(index=False, formatters=fmt))

## 3-regime HMM — does a third state add useful information?

In [ ]:
model3, labels3, scaler3 = fit_regime_model(macro, n_regimes=3)

summary3 = regime_summary(macro, labels3)
print("Regime summary (3 regimes):")
print(summary3.xs("mean", axis=1, level=1).round(2).to_string())
print(f"\nRegime counts:\n{labels3.value_counts().sort_index().to_string()}")

plot_regimes(macro, labels3, "3-regime HMM labels vs VIX (real macro data)")

## Save regime labels

In [ ]:
out = Path("../data/processed")
out.mkdir(parents=True, exist_ok=True)

labels2.to_frame().to_parquet(out / "regime_labels_2.parquet")
labels3.to_frame().to_parquet(out / "regime_labels_3.parquet")

# Default: 2-regime labels (used in notebook 06)
(labels2.to_frame()
 .rename_axis("date")
 .to_parquet(out / "regime_labels.parquet"))

print(f"Saved regime_labels.parquet, regime_labels_2.parquet, regime_labels_3.parquet")

## Are there signals that flip between regimes?

On **synthetic random data**, regime-conditional ICs are all near zero and the spread between regimes is noise — no signal is genuinely helpful in one regime and destructive in another, because the outcomes are random walks with no dependence on the macro state.

With **real equity data**, the story is more interesting. The 2-regime HMM reliably separates low-VIX trending markets (Regime 0) from high-VIX stressed markets (Regime 1), as reflected in the VIX and credit-spread means. In real data:

- **Momentum** tends to work in Regime 0 (trending, low vol) and reverse or fail in Regime 1 (crisis, mean-reversion dominates)
- **Short-term reversal** tends to strengthen in Regime 1 (overshoots and bounces are more pronounced in volatile markets)
- **Value (E/P, HML beta)** is more regime-neutral — it's a slow-moving signal that doesn't depend on short-term dynamics

If this pattern holds, regime-conditional combination has a reason to exist: upweight momentum in Regime 0, upweight reversal in Regime 1.

**On the 3-regime model:** a third state typically splits either the calm or the stressed regime into sub-states (e.g., a 'recovery' state between crisis and normal). Whether this adds value depends on whether the IC differences between the three regimes are large relative to estimation error. With short history and synthetic data, the third regime is likely fitting noise — the additional complexity isn't justified unless there's a clear economic interpretation for each state.